# 09 · LightGBM quantile blend — leak-free features, quarterly walk-forward, as-served validation

Notebook 06 trains a point-forecast LightGBM per cluster with `mlforecast`. On durable-goods revenue the
point forecast is rarely the number the business wants: the distribution is skewed (project deals, price
steps), the quarter total matters more than any month, and the forecast is made **at a quarter origin** for
the next three months with no actuals in between. This notebook is the modelling recipe that came out of a
long series of production experiments, generalised:

| ingredient | lesson it encodes |
|---|---|
| **one LightGBM per (cluster, quantile)**, grid `q10 … q95` | a point model cannot be moved up or down after the fact; a quantile grid can |
| **leak-free, origin-consistent lags** (`lag2`, `lag3`, `lag12`, no `lag1`) | when a quarter is forecast at its origin the previous month is unknown for months 2–3; lags that fall beyond the origin are replaced by the last known actual |
| **small, regularised trees for a single-series cluster** | large trees on 60 rows produce *crossing* quantiles (`q95 < q80` on a third of the rows); `num_leaves=3`, regularisation on, fixes it |
| **blend `w·q_lo + (1-w)·q_hi`** tuned per cluster by grid search on the **top-80 % revenue** series | the blend is a calibrated point forecast; tuning on every tiny series lets noise dominate |
| **oracle ceiling** (best quantile per row) | the floor no blend or rule can beat — tells you when to stop tuning |
| **readable vs strict (as-served) validation** | validation rows that carry the *actual* price / quantity change of the month make rule layers look good; rebuilt as served (extrapolated drivers, origin-consistent lags) the same rules can hurt. Decide on the strict number |
| **rules layer with a commit gate** | price / quantity rules are kept only where they beat both the plain blend *and* the seasonal naive **as served** |
| **scaled metrics next to WMAPE** (bias, median MASE, revenue-weighted MASE) | one yardstick across clusters; MASE grades a series against its own volatility |
| **folds anchored to the last data month** | validation = the last *N* complete fiscal quarters; served horizon = the rest of the current quarter (or the next one at a boundary) — never hard-coded dates |

| | |
|---|---|
| **Reads** | `<scenario>_clustered` (notebook 08) |
| **Writes** | `<scenario>_quantile_forecasts` — validation folds (as served) + the served quarter, with every quantile, the blend, the committed forecast and the seasonal naive; `<scenario>_quantile_config` — the per-cluster recipe |
| **Tuned clusters** | `strategic`, `smooth`, `erratic`; every other cluster is served with the seasonal naive (the Croston skill benchmarks the sparse ones) |

## 1. Imports

In [ ]:
import os
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

warnings.filterwarnings("ignore")
print("✅ Libraries imported | lightgbm", lgb.__version__)

## 2. Configuration

In [ ]:
SCENARIO = "durable_goods_monthly"
UID, DATE_VAR, TARGET, CLUSTER_COL = "unique_id", "ds", "y", "profile_cluster"
CATS = ["product_group", "region", "channel"]          # identity features (which series), categorical
SEASON = 12
RANDOM_STATE = 42

TUNE_CLUSTERS = ["strategic", "smooth", "erratic"]     # LightGBM quantile; everything else -> seasonal naive
QGRID = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
N_VAL_QUARTERS = 3                                     # complete fiscal quarters used for walk-forward validation
TOP_SHARE = 0.80                                       # tune on the series that make up 80% of revenue
BIAS_CAP = 0.15                                        # a blend whose net bias exceeds +-15% is not eligible

LGBM_BASE = {"objective": "quantile", "boosting_type": "gbdt", "verbose": -1, "n_jobs": -1, "random_state": RANDOM_STATE}
PARAMS = {
    # single large series: ~60 training rows -> tiny, regularised trees keep the quantiles monotonic
    "strategic": {"n_estimators": 250, "learning_rate": 0.05, "num_leaves": 3, "max_depth": 4, "min_child_samples": 10,
                  "reg_alpha": 0.5, "reg_lambda": 2.5, "subsample": 0.7, "subsample_freq": 1, "colsample_bytree": 0.9,
                  "min_split_gain": 0.05},
    # pooled clusters: many series -> deeper trees, regularisation on
    "smooth": {"n_estimators": 500, "learning_rate": 0.02, "num_leaves": 15, "min_child_samples": 20,
               "reg_alpha": 2.0, "reg_lambda": 10.0, "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8},
}
PARAMS["erratic"] = dict(PARAMS["smooth"])
BIG_TREE_PARAMS = {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 5}   # the anti-pattern

# rules layer candidates (only committed where they beat blend AND naive as served)
RULE_GRID = {"up_margin": [0.10, 0.25], "down_margin": [0.25, 0.50], "q_hi_rule": [0.80, 0.90], "q_lo_rule": [0.20, 0.30]}
EXTRAP_WINDOW = 6                                      # months of recent trend used for the served drivers

# IO
LAKEHOUSE_NAME = "ts_forecasting"
LOCAL_FOLDER = SCENARIO
root_path = Path.cwd().parent.parent
local_dir = root_path / "data" / LOCAL_FOLDER
CLUSTERED_TABLE = f"{SCENARIO}_clustered"
FORECAST_TABLE = f"{SCENARIO}_quantile_forecasts"
CONFIG_TABLE = f"{SCENARIO}_quantile_config"


def load_table(name):
    try:
        df = spark.table(f"{LAKEHOUSE_NAME}.{name}").toPandas()
        print(f"✅ Loaded {LAKEHOUSE_NAME}.{name} ({len(df):,} rows)")
    except Exception as e:
        df = pd.read_parquet(local_dir / f"{name}.parquet")
        print(f"ℹ️ Spark unavailable ({type(e).__name__}); loaded {local_dir / (name + '.parquet')} ({len(df):,} rows)")
    return df


def save_table(df, name):
    try:
        spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{LAKEHOUSE_NAME}.{name}")
        print(f"✅ Saved {len(df):,} rows to {LAKEHOUSE_NAME}.{name}")
    except Exception as e:
        local_dir.mkdir(parents=True, exist_ok=True)
        df.to_parquet(local_dir / f"{name}.parquet", index=False)
        print(f"ℹ️ Spark unavailable ({type(e).__name__}); saved {len(df):,} rows to {local_dir / (name + '.parquet')}")


print(f"✅ Config | tuned clusters {TUNE_CLUSTERS} | quantiles {QGRID} | {N_VAL_QUARTERS} validation quarters")

## 3. Load the clustered panel and build the feature frame

Features are deliberately few. **Not** included: `lag1` (unknown for months 2–3 of a quarter forecast at its
origin), moving averages and rolling stds (they embed `lag1`), and the price / quantity *levels* (they feed
the rules layer, not the model). `price_qoq` / `qty_qoq` are the change versus the same position one fiscal
quarter earlier.

In [ ]:
df = load_table(CLUSTERED_TABLE)
df[DATE_VAR] = pd.to_datetime(df[DATE_VAR])
df = df.sort_values([UID, DATE_VAR]).drop_duplicates([UID, DATE_VAR], keep="last").reset_index(drop=True)
FY_START_MONTH = int(df.loc[df["fiscal_month"] == 1, DATE_VAR].dt.month.iloc[0]) if "fiscal_month" in df.columns else 10
DATA_MAX = df[DATE_VAR].max()


def fiscal_parts(ts):
    fm = ((ts.month - FY_START_MONTH) % 12) + 1
    return ts.year + (1 if ts.month >= FY_START_MONTH else 0), (fm - 1) // 3 + 1, fm


def fq_label(ts):
    fy, fq, _ = fiscal_parts(ts)
    return f"FY{fy % 100:02d}-Q{fq}"


def calendar_features(dates):
    fm = ((dates.dt.month - FY_START_MONTH) % 12) + 1
    return pd.DataFrame({"fiscal_month": fm, "fiscal_quarter": (fm - 1) // 3 + 1,
                         "quarter_end": fm.isin([3, 6, 9, 12]).astype(int), "fy_end": (fm == 12).astype(int),
                         "days_in_month": dates.dt.days_in_month}, index=dates.index)


CAL_FEATURES = ["fiscal_month", "fiscal_quarter", "quarter_end", "fy_end", "days_in_month"]
LAG_FEATURES = ["y_lag2", "y_lag3", "y_lag12"]
MODEL_FEATURES = CAL_FEATURES + LAG_FEATURES + CATS
CAT_LEVELS = {c: sorted(df[c].dropna().astype(str).unique()) for c in CATS}


def build_features(panel):
    f = panel.sort_values([UID, DATE_VAR]).copy()
    f[CAL_FEATURES] = calendar_features(f[DATE_VAR])
    g = f.groupby(UID, sort=False)
    for k in (2, 3, 12):
        f[f"y_lag{k}"] = g[TARGET].shift(k)
    if "price_per_unit" not in f.columns:
        f["price_per_unit"] = f[TARGET] / f["quantity"].replace(0, np.nan)
    f["price_qoq"] = g["price_per_unit"].pct_change(3)
    f["qty_qoq"] = g["quantity"].pct_change(3)
    for c in CATS:
        f[c] = pd.Categorical(f[c].astype(str), categories=CAT_LEVELS[c])
    return f


feat = build_features(df)
ALL_CLUSTERS = sorted(feat[CLUSTER_COL].dropna().unique())
NAIVE_CLUSTERS = [c for c in ALL_CLUSTERS if c not in TUNE_CLUSTERS]
print(f"feature frame {feat.shape} | data through {DATA_MAX.date()} ({fq_label(DATA_MAX)}) | FY starts month {FY_START_MONTH}")
print(f"model features ({len(MODEL_FEATURES)}): {MODEL_FEATURES}")
print(f"naive clusters: {NAIVE_CLUSTERS}")

## 4. Walk-forward folds anchored to the last data month

Validation = the last `N_VAL_QUARTERS` **complete** fiscal quarters on or before `DATA_MAX`, each forecast
from an origin one month before the quarter starts. The served horizon is the rest of the current quarter
when `DATA_MAX` is month 1 or 2 of a quarter, or the whole next quarter when it is month 3. Exactly one
served fold always exists, in any month of any fiscal year.

In [ ]:
def month_in_quarter(ts):
    return ((fiscal_parts(ts)[2] - 1) % 3) + 1


def quarter_start(ts):
    return ts.replace(day=1) - pd.DateOffset(months=month_in_quarter(ts) - 1)


def quarter_months(q_start):
    return [q_start + pd.DateOffset(months=i) for i in range(3)]


def make_folds(data_max, n_val):
    cur = quarter_start(data_max)
    miq = month_in_quarter(data_max)
    last_complete = cur if miq == 3 else cur - pd.DateOffset(months=3)
    folds = []
    for i in range(n_val, 0, -1):
        qs = last_complete - pd.DateOffset(months=3 * (i - 1))
        folds.append({"name": f"fold{n_val - i + 1}", "quarter": fq_label(qs), "origin": qs - pd.DateOffset(months=1),
                      "months": quarter_months(qs), "future": False})
    if miq == 3:
        served = quarter_months(cur + pd.DateOffset(months=3))
    else:
        served = [m for m in quarter_months(cur) if m > data_max]
    folds.append({"name": "final", "quarter": fq_label(served[0]), "origin": data_max, "months": served, "future": True})
    return folds


FOLDS = make_folds(DATA_MAX, N_VAL_QUARTERS)
VAL_FOLDS = [f for f in FOLDS if not f["future"]]
FINAL_FOLD = [f for f in FOLDS if f["future"]][0]
for f in FOLDS:
    print(f"  {f['name']:6s} | {f['quarter']} | train <= {f['origin'].date()} -> forecast {[m.strftime('%Y-%m') for m in f['months']]}"
          + ("  (served; booked months of the quarter carry actuals)" if f["future"] else ""))

## 5. Engine — training, as-served rows, forecast frames

`served_rows()` is the heart of the strict validation: for every target month it rebuilds the row **exactly
as production will see it** — identity from the last known row, calendar of the target month, lags that fall
beyond the origin replaced by the last known actual, price / quantity extended with the median step of the
last `EXTRAP_WINDOW` months. The *readable* frame instead takes the rows as they exist in the feature table
(contemporaneous drivers, lags from the realised history).

In [ ]:
def qcol(q):
    return f"q{int(round(q * 100))}"


def _lgbm(cluster, q):
    return lgb.LGBMRegressor(**{**LGBM_BASE, "alpha": float(np.clip(q, 1e-3, 0.999)), **PARAMS.get(cluster, PARAMS["smooth"])})


def train_cluster_quantiles(origin, qgrid, params_override=None, weight_fn=None):
    """One LightGBM per (tuned cluster, quantile) on rows <= origin. weight_fn(train_df, origin) -> sample weights."""
    models = {}
    for clu in TUNE_CLUSTERS:
        tr = feat[(feat[CLUSTER_COL] == clu) & (feat[DATE_VAR] <= origin) & feat[TARGET].notna()].sort_values([UID, DATE_VAR])
        if tr.empty:
            continue
        X, yv = tr[MODEL_FEATURES], tr[TARGET]
        w = None
        if weight_fn is not None:
            w = np.asarray(weight_fn(tr, origin), float)
            w = w / w.mean()                       # quantile objective: keep sample weights on a mean-1 scale
        models[clu] = {}
        for q in qgrid:
            m = _lgbm(clu, q) if params_override is None else lgb.LGBMRegressor(**{**LGBM_BASE, "alpha": q, **params_override})
            models[clu][q] = m.fit(X, yv, sample_weight=w, categorical_feature=CATS)
    return models


def _extrapolate(series, upto):
    s = series.dropna().sort_index()
    if s.empty:
        return pd.Series(dtype=float)
    diffs = np.diff(s.iloc[-EXTRAP_WINDOW:].to_numpy(float))
    step = float(np.median(diffs)) if len(diffs) else 0.0
    idx = pd.date_range(s.index.min(), max(s.index.max(), upto), freq="MS")
    path = s.reindex(idx)
    cur = float(s.iloc[-1])
    for m in idx[idx > s.index.max()]:
        cur = max(cur + step, 0.0)
        path.loc[m] = cur
    return path


def served_rows(cdf, months, origin):
    """As-served rows for every series of a cluster frame: origin-consistent lags + extrapolated drivers."""
    hist = cdf[(cdf[DATE_VAR] <= origin) & cdf[TARGET].notna()]
    out = []
    for uid, g in hist.groupby(UID, sort=False, observed=True):
        g = g.sort_values(DATE_VAR)
        yv = g.set_index(DATE_VAR)[TARGET]
        last_y, last_row = float(yv.iloc[-1]), g.iloc[-1]
        price = _extrapolate(g.set_index(DATE_VAR)["price_per_unit"], max(months))
        qty = _extrapolate(g.set_index(DATE_VAR)["quantity"], max(months))
        for m in months:
            lag = lambda k: float(yv.get(m - pd.DateOffset(months=k), last_y)) if (m - pd.DateOffset(months=k)) <= origin else last_y
            def qoq(p):
                a, b = p.get(m, np.nan), p.get(m - pd.DateOffset(months=3), np.nan)
                return (a - b) / abs(b) if (pd.notna(a) and pd.notna(b) and b != 0) else np.nan
            out.append({UID: uid, DATE_VAR: m, CLUSTER_COL: last_row[CLUSTER_COL],
                        **{c: last_row[c] for c in CATS},
                        "y_lag2": lag(2), "y_lag3": lag(3), "y_lag12": lag(12),
                        "price_per_unit": price.get(m, np.nan), "quantity": qty.get(m, np.nan),
                        "price_qoq": qoq(price), "qty_qoq": qoq(qty),
                        "snaive": float(yv.get(m - pd.DateOffset(months=SEASON), last_y))})
    rows = pd.DataFrame(out)
    if rows.empty:
        return rows
    rows[CAL_FEATURES] = calendar_features(rows[DATE_VAR])
    for c in CATS:
        rows[c] = pd.Categorical(rows[c].astype(str), categories=CAT_LEVELS[c])
    return rows


def build_frame(origin, months, models, mode, rearrange=True):
    """Quantile predictions for every tuned-cluster series x target month. mode: 'readable' | 'strict'.
    rearrange=True sorts the predicted quantiles within each row (quantile rearrangement) so they never cross."""
    actual = feat.set_index([UID, DATE_VAR])[TARGET]
    parts = []
    for clu, qmods in models.items():
        cdf = feat[feat[CLUSTER_COL] == clu]
        if mode == "readable":
            block = cdf[cdf[DATE_VAR].isin(months)].copy()
            block["snaive"] = [actual.get((u, d - pd.DateOffset(months=SEASON)), np.nan) for u, d in zip(block[UID], block[DATE_VAR])]
        else:
            block = served_rows(cdf, months, origin)
        if block.empty:
            continue
        block["y"] = [actual.get((u, d), np.nan) for u, d in zip(block[UID], block[DATE_VAR])]
        qs = sorted(qmods)
        for q in qs:
            block[qcol(q)] = qmods[q].predict(block[MODEL_FEATURES])
        if rearrange:
            cols = [qcol(q) for q in qs]
            block[cols] = np.sort(block[cols].to_numpy(float), axis=1)
        parts.append(block)
    fc = pd.concat(parts, ignore_index=True)
    fc["origin"], fc["mode"] = origin, mode
    fc["has_actual"] = fc["y"].notna()
    return fc


def seasonal_naive_all(months):
    """Seasonal naive for EVERY series (used for the naive clusters and as the baseline)."""
    actual = feat.set_index([UID, DATE_VAR])[TARGET]
    last = feat.sort_values(DATE_VAR).groupby(UID, observed=True)[TARGET].last()
    rows = [{UID: u, DATE_VAR: m, "snaive": max(float(actual.get((u, m - pd.DateOffset(months=SEASON)), last[u])), 0.0)}
            for u in last.index for m in months]
    return pd.DataFrame(rows)


print("✅ engine ready")

## 6. Train every validation fold once; build the readable and the strict frame

In [ ]:
FOLD_MODELS, FRAMES = {}, {"readable": {}, "strict": {}}
for f in VAL_FOLDS:
    FOLD_MODELS[f["name"]] = train_cluster_quantiles(f["origin"], QGRID)
    for mode in FRAMES:
        FRAMES[mode][f["name"]] = build_frame(f["origin"], f["months"], FOLD_MODELS[f["name"]], mode)
    n_s = int(FRAMES["strict"][f["name"]]["has_actual"].sum())
    print(f"  {f['name']}: trained {sum(len(v) for v in FOLD_MODELS[f['name']].values())} quantile models | {n_s} scored rows (strict)")

# revenue-focus set: the series that make up TOP_SHARE of revenue (tuning objective)
_rev = feat.groupby(UID, observed=True)[TARGET].sum().sort_values(ascending=False)
_cum = _rev.cumsum() / _rev.sum()
FOCUS = set(_rev.index[(_cum.shift(fill_value=0.0) < TOP_SHARE)])
print(f"revenue-focus set: {len(FOCUS)} series cover {TOP_SHARE:.0%} of revenue")

## 7. Metrics — one yardstick across clusters

- **WMAPE** = Σ|error| / Σ|actual| (pooled dollars; the gate metric).
- **bias %** = Σ(forecast − actual) / Σ|actual| (net over / under-forecast — a WMAPE-optimal blend of a lumpy
  cluster is often 30–40 % *low*; the bias cap keeps it honest).
- **MASE** = per-series MAE ÷ that series' mean |month-to-month change| over its training history. `< 1`
  beats a one-step naive on *that* series. Reported as the **median** across series (typical series) and
  **revenue-weighted** (`wMASE`, the big series). The plain mean is unusable — a near-zero series can post a
  MASE in the thousands.

In [ ]:
def wmape(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    m = ~np.isnan(a) & ~np.isnan(p)
    d = np.abs(a[m]).sum()
    return np.nan if d == 0 else 100.0 * np.abs(a[m] - p[m]).sum() / d


def bias_pct(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    m = ~np.isnan(a) & ~np.isnan(p)
    d = np.abs(a[m]).sum()
    return np.nan if d == 0 else 100.0 * (p[m] - a[m]).sum() / d


def naive_scale(origin):
    h = feat[(feat[DATE_VAR] <= origin) & feat[TARGET].notna()].sort_values([UID, DATE_VAR])
    return h.groupby(UID, observed=True)[TARGET].diff().abs().groupby(h[UID], observed=True).mean()


def scaled_metrics(frame, col):
    s = frame[frame["y"].notna() & frame[col].notna()]
    if s.empty:
        return {}
    out = {"n_rows": len(s), "wmape": wmape(s["y"], s[col]), "bias_pct": bias_pct(s["y"], s[col])}
    sc = pd.concat([naive_scale(o).rename("scale") for o in s["origin"].unique()], axis=1).mean(axis=1)
    per = (pd.DataFrame({UID: s[UID].astype(str).values, "ae": np.abs(s[col] - s["y"]).values, "ay": np.abs(s["y"]).values})
           .groupby(UID).agg(mae=("ae", "mean"), ay=("ay", "sum")))
    per["mase"] = per["mae"] / per.index.map(sc.astype(float).to_dict()).astype(float)
    per = per.replace([np.inf, -np.inf], np.nan).dropna()
    if len(per):
        w = per["ay"] / per["ay"].sum()
        out.update(mase_med=float(per["mase"].median()), wmase=float((per["mase"] * w).sum()), n_series=len(per))
    return out


print("✅ metric helpers ready")

## 8. Quantile crossing — why the single-series cluster needs tiny trees, and the rearrangement repair

Quantile regressors are fitted independently, so nothing forces `q90 ≥ q80`. On a cluster with one series
(~60 rows) large, unregularised trees cross on a large share of adjacent pairs and the upper quantiles — the
ones the business wants when the market steps up — collapse towards the median. Two complementary answers:

1. **the recipe** (tiny, regularised trees) — fewer crossings *and* a wider, more honest upper band;
2. **quantile rearrangement** (sort the predicted quantiles within each row) — removes the remaining crossings
   and never worsens the pinball loss; it is a repair, not a substitute for (1): rearranging a too-narrow band
   still leaves it too narrow. Every frame used downstream is rearranged.

Measured on the first validation fold, as served.

In [ ]:
def crossing_rate(frame, qgrid=QGRID):
    P = frame[[qcol(q) for q in qgrid]].to_numpy(float)
    return float((np.diff(P, axis=1) < -1e-9).mean()) if len(P) else np.nan


def band_ratio(frame):
    return float((frame["q90"] / frame["q50"].replace(0, np.nan)).median())


f1 = VAL_FOLDS[0]
raw_recipe = build_frame(f1["origin"], f1["months"], FOLD_MODELS[f1["name"]], "strict", rearrange=False)
big = train_cluster_quantiles(f1["origin"], QGRID, params_override=BIG_TREE_PARAMS)
raw_big = build_frame(f1["origin"], f1["months"], {"strategic": big["strategic"]}, "strict", rearrange=False)
rows = []
for clu in TUNE_CLUSTERS:
    t = raw_recipe[raw_recipe[CLUSTER_COL] == clu]
    rows.append({"cluster": clu, "params": "recipe (raw)", "crossing_rate": crossing_rate(t), "q90/q50 median": band_ratio(t)})
rows.append({"cluster": "strategic", "params": "big trees, no regularisation (raw)", "crossing_rate": crossing_rate(raw_big), "q90/q50 median": band_ratio(raw_big)})
t = FRAMES["strict"][f1["name"]]
rows.append({"cluster": "strategic", "params": "recipe + rearrangement (used downstream)",
             "crossing_rate": crossing_rate(t[t[CLUSTER_COL] == "strategic"]), "q90/q50 median": band_ratio(t[t[CLUSTER_COL] == "strategic"])})
print(pd.DataFrame(rows).round(3).to_string(index=False))

## 9. Blend tuning per cluster, oracle ceiling, and the readable-vs-strict gap

For each tuned cluster the grid `(q_lo, q_hi, w)` is searched on the **strict** frames pooled over the
validation folds, restricted to the revenue-focus series, minimising WMAPE under the bias cap. The oracle
(best quantile per row) is the ceiling. The same blend is then scored on the readable frames to show how
much an optimistic validation can overstate production accuracy.

In [ ]:
def apply_blend(fc, blend):
    fc = fc.copy()
    fc["yhat_blend"] = blend["w"] * fc[qcol(blend["q_lo"])] + (1.0 - blend["w"]) * fc[qcol(blend["q_hi"])]
    return fc


def pooled(frames, clu, col, focus_only=True):
    parts = [f[(f[CLUSTER_COL] == clu) & f["has_actual"] & (f[UID].astype(str).isin(FOCUS) if focus_only else True)] for f in frames]
    s = pd.concat(parts, ignore_index=True)
    return (wmape(s["y"], s[col]), bias_pct(s["y"], s[col])) if len(s) else (np.nan, np.nan)


def oracle_wmape(frames, clu):
    s = pd.concat([f[(f[CLUSTER_COL] == clu) & f["has_actual"]] for f in frames], ignore_index=True)
    P = s[[qcol(q) for q in QGRID]].to_numpy(float)
    A = s["y"].to_numpy(float)
    best = P[np.arange(len(s)), np.argmin(np.abs(P - A[:, None]), axis=1)]
    return wmape(A, best)


BLEND_GRID = [(lo, hi, w) for lo in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7) for hi in (0.6, 0.7, 0.8, 0.9, 0.95) if hi > lo
              for w in (0.2, 0.35, 0.5, 0.65, 0.8)]
strict_frames = [FRAMES["strict"][f["name"]] for f in VAL_FOLDS]
readable_frames = [FRAMES["readable"][f["name"]] for f in VAL_FOLDS]
BEST_BLEND, summary = {}, []
for clu in TUNE_CLUSTERS:
    cands = []
    for lo, hi, w in BLEND_GRID:
        b = {"q_lo": lo, "q_hi": hi, "w": w}
        e, bias = pooled([apply_blend(f, b) for f in strict_frames], clu, "yhat_blend")
        cands.append({**b, "wmape": e, "bias": bias, "eligible": abs(bias) <= BIAS_CAP * 100})
    cands = pd.DataFrame(cands).sort_values("wmape")
    pick = cands[cands["eligible"]].iloc[0] if cands["eligible"].any() else cands.iloc[0]
    BEST_BLEND[clu] = {"q_lo": float(pick["q_lo"]), "q_hi": float(pick["q_hi"]), "w": float(pick["w"])}
    r_e, r_b = pooled([apply_blend(f, BEST_BLEND[clu]) for f in readable_frames], clu, "yhat_blend")
    n_e, n_b = pooled(strict_frames, clu, "snaive")
    summary.append({"cluster": clu, "blend": f"{pick['w']:.2f}*q{int(pick['q_lo']*100)} + {1-pick['w']:.2f}*q{int(pick['q_hi']*100)}",
                    "strict_wmape": pick["wmape"], "strict_bias": pick["bias"], "readable_wmape": r_e,
                    "unconstrained_wmape": cands.iloc[0]["wmape"], "unconstrained_bias": cands.iloc[0]["bias"],
                    "snaive_wmape": n_e, "oracle_wmape": oracle_wmape(strict_frames, clu)})
blend_summary = pd.DataFrame(summary)
print("Blend tuned on the STRICT (as-served) frames, revenue-focus series, bias cap +-%d%%:" % int(BIAS_CAP * 100))
print(blend_summary.round(1).to_string(index=False))

## 10. Rules layer with an as-served commit gate

**Up rule**: when the price *or* quantity driver rises by more than `up_margin` (and neither falls by more
than it) the forecast is lifted to `q_hi_rule`. **Down rule**: when quantity falls by `down_margin` or more
the forecast drops to `q_lo_rule`. In the readable frame the drivers are the month's *actual* changes — the
rule is grading itself with the answer key. As served the drivers are median-step extrapolations, so a rising
trend fires the up rule month after month. The commit gate therefore reads the **strict** column: rules are
kept only where they beat both the plain blend and the seasonal naive as served.

In [ ]:
def apply_rules(fc, rules):
    fc = fc.copy()
    pq, qq = fc["price_qoq"], fc["qty_qoq"]
    up = (((pq > rules["up_margin"]) | (qq > rules["up_margin"])) & ~((pq < -rules["up_margin"]) | (qq < -rules["up_margin"]))).fillna(False)
    dn = (qq <= -rules["down_margin"]).fillna(False) & ~up
    fc["yhat_rules"] = fc["yhat_blend"]
    fc.loc[up, "yhat_rules"] = fc.loc[up, qcol(rules["q_hi_rule"])]
    fc.loc[dn, "yhat_rules"] = fc.loc[dn, qcol(rules["q_lo_rule"])]
    fc["rule_fired"] = np.where(up, "up", np.where(dn, "down", "none"))
    return fc


COMMITTED, gate_rows = {}, []
for clu in TUNE_CLUSTERS:
    b = BEST_BLEND[clu]
    best_rules, best_strict = None, np.inf
    for um in RULE_GRID["up_margin"]:
        for dm in RULE_GRID["down_margin"]:
            for qh in RULE_GRID["q_hi_rule"]:
                for ql in RULE_GRID["q_lo_rule"]:
                    rules = {"up_margin": um, "down_margin": dm, "q_hi_rule": qh, "q_lo_rule": ql}
                    e, _ = pooled([apply_rules(apply_blend(f, b), rules) for f in strict_frames], clu, "yhat_rules")
                    if e < best_strict:
                        best_rules, best_strict = rules, e
    e_blend_s, _ = pooled([apply_blend(f, b) for f in strict_frames], clu, "yhat_blend")
    e_naive_s, _ = pooled(strict_frames, clu, "snaive")
    e_rules_r, _ = pooled([apply_rules(apply_blend(f, b), best_rules) for f in readable_frames], clu, "yhat_rules")
    e_blend_r, _ = pooled([apply_blend(f, b) for f in readable_frames], clu, "yhat_blend")
    fired = pd.concat([apply_rules(apply_blend(f, b), best_rules) for f in strict_frames])
    fired = fired[(fired[CLUSTER_COL] == clu) & fired["has_actual"]]
    commit_rules = best_strict < e_blend_s and best_strict < e_naive_s
    method = "blend+rules" if commit_rules else ("blend" if e_blend_s <= e_naive_s else "seasonal_naive")
    COMMITTED[clu] = {"method": method, "blend": b, "rules": best_rules if commit_rules else None}
    gate_rows.append({"cluster": clu, "snaive": e_naive_s, "blend_readable": e_blend_r, "blend_strict": e_blend_s,
                      "rules_readable": e_rules_r, "rules_strict": best_strict,
                      "rule_fired_share_strict": float((fired["rule_fired"] != "none").mean()), "committed": method})
gate = pd.DataFrame(gate_rows)
print("Commit gate (WMAPE %, revenue-focus series): rules are committed only when rules_strict < blend_strict AND < snaive")
print(gate.round(1).to_string(index=False))

## 11. Scaled accuracy per cluster × validation quarter (as served)

In [ ]:
def committed_frame(fc):
    out = []
    for clu, cfg in COMMITTED.items():
        s = apply_blend(fc[fc[CLUSTER_COL] == clu], cfg["blend"])
        if cfg["rules"]:
            s = apply_rules(s, cfg["rules"])
        else:
            s["yhat_rules"], s["rule_fired"] = s["yhat_blend"], "none"
        s["yhat_committed"] = {"blend+rules": s["yhat_rules"], "blend": s["yhat_blend"], "seasonal_naive": s["snaive"]}[cfg["method"]]
        s["cluster_method"] = cfg["method"]
        out.append(s)
    return pd.concat(out, ignore_index=True)


VAL_FC = {f["name"]: committed_frame(FRAMES["strict"][f["name"]]) for f in VAL_FOLDS}
rows = []
for f in VAL_FOLDS:
    for clu, g in VAL_FC[f["name"]].groupby(CLUSTER_COL, observed=True):
        for col in ("yhat_committed", "snaive"):
            m = scaled_metrics(g, col)
            if m:
                rows.append({"cluster": clu, "quarter": f["quarter"], "forecast": col.replace("yhat_", ""), **m})
_all = pd.concat(VAL_FC.values(), ignore_index=True)
for clu, g in _all.groupby(CLUSTER_COL, observed=True):
    for col in ("yhat_committed", "snaive"):
        rows.append({"cluster": clu, "quarter": "ALL FOLDS", "forecast": col.replace("yhat_", ""), **scaled_metrics(g, col)})
for col in ("yhat_committed", "snaive"):
    rows.append({"cluster": "ALL TUNED", "quarter": "ALL FOLDS", "forecast": col.replace("yhat_", ""), **scaled_metrics(_all, col)})
metrics_df = pd.DataFrame(rows).reindex(columns=["cluster", "quarter", "forecast", "n_series", "n_rows", "wmape", "bias_pct", "mase_med", "wmase"])
print(metrics_df.round(2).to_string(index=False))

## 12. Serve the current quarter — every series, full quantile band

Final models are trained on **all** data through `DATA_MAX` (validation folds train on subsets to *measure*;
the served model must see the latest months). Tuned clusters get the committed method; every other series
gets the seasonal naive; months of the served quarter that are already booked carry the actual, so the quarter
always sums to a full quarter.

In [ ]:
FINAL_MODELS = train_cluster_quantiles(DATA_MAX, QGRID)
served_q_months = quarter_months(quarter_start(FINAL_FOLD["months"][0]))
fut = committed_frame(build_frame(DATA_MAX, served_q_months, FINAL_MODELS, "strict"))
naive = seasonal_naive_all(served_q_months)
naive = naive[~naive[UID].astype(str).isin(set(fut[UID].astype(str)))].copy()
static = feat.sort_values(DATE_VAR).groupby(UID, observed=True)[[CLUSTER_COL] + CATS].last().reset_index()
naive = naive.merge(static, on=UID, how="left")
naive["yhat_blend"] = naive["yhat_rules"] = naive["yhat_committed"] = naive["snaive"]
naive["rule_fired"], naive["cluster_method"], naive["origin"], naive["mode"] = "naive", "seasonal_naive", DATA_MAX, "strict"
actual = feat.set_index([UID, DATE_VAR])[TARGET]
naive["y"] = [actual.get((u, d), np.nan) for u, d in zip(naive[UID], naive[DATE_VAR])]
naive["has_actual"] = naive["y"].notna()
served = pd.concat([fut, naive], ignore_index=True)
served["is_future"] = served[DATE_VAR] > DATA_MAX
booked = ~served["is_future"] & served["y"].notna()
for c in ("yhat_blend", "yhat_rules", "yhat_committed", "snaive"):
    served.loc[booked, c] = served.loc[booked, "y"]
served["fold"], served["quarter"] = "final", FINAL_FOLD["quarter"]

tot = served.groupby("is_future")[["yhat_committed", "snaive"]].sum() / 1e6
print(f"served quarter {FINAL_FOLD['quarter']}: {served[UID].nunique()} series x {len(served_q_months)} months "
      f"({int((~served['is_future']).sum() / served[UID].nunique())} booked month(s))")
print(f"  committed forecast, forecast months: {tot.loc[True, 'yhat_committed']:.1f} M | seasonal naive {tot.loc[True, 'snaive']:.1f} M")
print(served[served["is_future"]].groupby(CLUSTER_COL, observed=True)
      .agg(series=(UID, "nunique"), committed_M=("yhat_committed", lambda s: s.sum() / 1e6), snaive_M=("snaive", lambda s: s.sum() / 1e6),
           method=("cluster_method", "first")).round(1).to_string())

## 13. Charts — anchored actual vs forecast per fold and per cluster, served quarter with its band

In [ ]:
def cluster_hist(clu):
    d = feat if clu == "ALL" else feat[feat[CLUSTER_COL] == clu]
    return d.groupby(DATE_VAR)[TARGET].sum()


fig, axes = plt.subplots(len(TUNE_CLUSTERS), 1, figsize=(12, 3.4 * len(TUNE_CLUSTERS)), sharex=True)
palette = ["#1f77b4", "#2ca02c", "#9467bd"]
for ax, clu in zip(axes, TUNE_CLUSTERS):
    h = cluster_hist(clu) / 1e6
    h = h[h.index >= DATA_MAX - pd.DateOffset(months=30)]
    ax.plot(h.index, h.values, color="black", linewidth=1.6, label="actual")
    for k, f in enumerate(VAL_FOLDS):
        g = VAL_FC[f["name"]]
        g = g[g[CLUSTER_COL] == clu].groupby(DATE_VAR)[["yhat_committed", "snaive"]].sum() / 1e6
        anchor_m = h.index[h.index <= f["origin"]].max()
        x = [anchor_m] + list(g.index)
        ax.plot(x, [h.loc[anchor_m]] + list(g["yhat_committed"]), "o-", color=palette[k % 3], ms=4, label=f"{f['name']} committed")
        ax.plot(x, [h.loc[anchor_m]] + list(g["snaive"]), "^:", color=palette[k % 3], ms=4, alpha=0.6, label=f"{f['name']} seasonal naive")
    s = served[served[CLUSTER_COL] == clu]
    gs = s.groupby(DATE_VAR).agg(committed=("yhat_committed", "sum"), lo=("q10", "sum"), hi=("q90", "sum")) / 1e6
    ax.plot(gs.index, gs["committed"], "s--", color="#d62728", label="served committed")
    ax.fill_between(gs.index, gs["lo"], gs["hi"], color="#d62728", alpha=0.12, label="served q10-q90 (summed)")
    ax.axvline(DATA_MAX, color="red", linestyle=":", linewidth=1)
    ax.set_title(f"{clu} - committed method: {COMMITTED[clu]['method']}", loc="left", fontsize=10)
    ax.set_ylabel("revenue (M)")
axes[0].legend(fontsize=7, ncol=4)
fig.tight_layout()
plt.show()

## 14. Persist the forecasts and the recipe

In [ ]:
KEEP = [UID, DATE_VAR, CLUSTER_COL, "fold", "quarter", "origin", "is_future", "has_actual", "y", "yhat_blend", "yhat_rules",
        "yhat_committed", "snaive", "rule_fired", "cluster_method", "price_qoq", "qty_qoq"] + [qcol(q) for q in QGRID]
val_out = []
for f in VAL_FOLDS:
    v = VAL_FC[f["name"]].copy()
    v["fold"], v["quarter"], v["is_future"] = f["name"], f["quarter"], False
    val_out.append(v)
forecasts = pd.concat(val_out + [served], ignore_index=True)
for c in KEEP:
    if c not in forecasts.columns:
        forecasts[c] = np.nan
forecasts = forecasts[KEEP].copy()
forecasts[CLUSTER_COL] = forecasts[CLUSTER_COL].astype(str)
forecasts["origin"] = pd.to_datetime(forecasts["origin"])
save_table(forecasts, FORECAST_TABLE)

config = {"scenario": SCENARIO, "data_through": str(DATA_MAX.date()), "fy_start_month": FY_START_MONTH, "qgrid": QGRID,
          "model_features": MODEL_FEATURES, "params": PARAMS, "committed": COMMITTED,
          "validation": [{"name": f["name"], "quarter": f["quarter"], "origin": str(f["origin"].date())} for f in VAL_FOLDS],
          "served": {"quarter": FINAL_FOLD["quarter"], "months": [m.strftime("%Y-%m") for m in FINAL_FOLD["months"]]},
          "gate": gate.round(2).to_dict(orient="records"), "blend_summary": blend_summary.round(2).to_dict(orient="records")}
save_table(pd.DataFrame([{"key": "config", "json": json.dumps(config, default=str)}]), CONFIG_TABLE)
print(json.dumps(COMMITTED, indent=2, default=str))

## ✅ CHECK POINT with the data scientist

1. **Crossing table** — the single-series cluster must show a near-zero crossing rate; if not, shrink the trees
   before touching anything else.
2. **Blend table** — compare `strict_wmape` with `readable_wmape` and with `snaive_wmape`; the number to quote is
   the strict one. The `oracle_wmape` column is the floor: when the committed forecast is within a few points of
   it, further blend / rule tuning cannot help — only a different model family (or a different aggregation level,
   see notebook 11) can.
3. **Commit gate** — rules that look good in the readable column and bad in the strict column are firing on
   extrapolated trends, not evidence; leave them off.
4. **Bias** — a committed forecast with a large negative bias on a lumpy cluster under-forecasts the total even
   when its WMAPE is the best; the cap is a business decision, make it explicit.

**Next:** [10 · Recency weighting with a surge gate](10-Recency-Weighting-Surge-Gate) tests whether weighting
recent months helps the clusters whose level stepped up; [11 · Aggregate anchor](11-Aggregate-Anchor-Reconciliation)
reconciles this bottom-up forecast to a quarterly model at the product-family level.